In [21]:
import os
import pandas as pd
import json
import statsmodels.api as sm
from statsmodels.formula.api import ols
import scipy.stats as stats

## Loading the data

In [29]:
def get_results_from_benchmark_file(filename):
    with open(filename, "r") as file:
        response = json.load(file)
    
    result = response["scores"]

    # I mixed up the prompt type and user prompt when writing to the files
    result["user_prompt"] = response["prompt_type"]
    result["generator"] = response["generator"]
    result["prompt_type"] = response["user_prompt"]
    return result

In [30]:
"""Load benchmark"""
output_file = f"../results/result_eval_benchmark.json"
if os.path.exists(output_file):
    os.remove(output_file)

results = []

for generator_folder in os.listdir(f"../results/generation_only"):
            if generator_folder.startswith("."):
                continue

            for result_file in os.listdir(
                f"../results/generation_only/{generator_folder}"
            ):
                if result_file.startswith("."):
                    continue

                results.append(get_results_from_benchmark_file(
                    f"../results/generation_only/{generator_folder}/{result_file}"
                ))

with open(f"../responses/result_eval_benchmark.json", "w") as benchmark_file:
    json.dump(results, benchmark_file)

benchmark_df = pd.DataFrame(results)
benchmark_df

,answer_relevancy,factual_correctness,bleu,user_prompt,generator,prompt_type
0,0.258086,0.75,0.030162,Wat kun je me vertellen over Rick Kruys?,Ministral-3-14B-Instruct-2512,simple_question
1,0.450801,NaN,0.039195,Wat kun je me vertellen over Gemeente Amsterdam?,Ministral-3-14B-Instruct-2512,simple_question
2,0.338714,NaN,0.024387,Wat kun je me vertellen over Texel?,Ministral-3-14B-Instruct-2512,simple_question
3,0.303059,NaN,0.026227,Wat kun je me vertellen over Bruinvis?,Ministral-3-14B-Instruct-2512,simple_question
4,0.308608,NaN,0.020706,Wat kun je me vertellen over het project Zuida...,Ministral-3-14B-Instruct-2512,context_question
...,...,...,...,...,...,...
259,0.233125,NaN,0.025786,Bruinvis,Qwen3-30B-A3B-Instruct-2507,single_term
260,0.306629,NaN,0.072893,Wat kun je me vertellen over Trekkertrek?,Qwen3-30B-A3B-Instruct-2507,simple_question
261,0.000000,NaN,0.061098,Ik ben een journalist uit Noord-Holland en ik ...,Qwen3-30B-A3B-Instruct-2507,journalist_question
262,0.403136,NaN,0.023542,Wat kun je me vertellen over de Gemeente Amste...,Qwen3-30B-A3B-Instruct-2507,context_question


In [ ]:
"""Use this when there is a file containing the results directly."""

with open(f"../results/result_eval.json", 'r') as result_file:
    results = json.load(result_file)
    
results_df = pd.DataFrame(results)
multi_index_results_df = results_df.set_index(['embedder', 'generator', 'distance_metric']).sort_index()
multi_index_results_df

In [ ]:
def get_results_from_result_file(filename):
    with open(filename, "r") as file:
        response = json.load(file)
    
    result = response["scores"]
    result["user_prompt"] = response["user_prompt"]
    result["embedder"] = response["embedder"]
    result["distance_metric"] = response["distance_metric"]
    result["generator"] = response["generator"]
    result["prompt_type"] = response["prompt_type"]
    return result

In [ ]:
"""Use this when you want to calculate results based on the contents of the individual files."""

output_file = f"../results/result_eval.json"
if os.path.exists(output_file):
    os.remove(output_file)

results = []

for embedder_folder in os.listdir("../results"):
    if embedder_folder.startswith(".") or embedder_folder == "generation_only":
        continue

    if embedder_folder == "BM25":
        for generator_folder in os.listdir(f"../results/{embedder_folder}"):

            if generator_folder.startswith("."):
                continue

            for result_file in os.listdir(
                f"../results/{embedder_folder}/{generator_folder}"
            ):
                if result_file.startswith("."):
                    continue

                results.append(get_results_from_result_file(f"../results/{embedder_folder}/{generator_folder}/{result_file}"))

    else:
        for generator_folder in os.listdir(f"../results/{embedder_folder}"):
            if generator_folder.startswith("."):
                continue

            for distance_metric in os.listdir(
                f"../results/{embedder_folder}/{generator_folder}"
            ):
                if distance_metric.startswith("."):
                    continue

                for result_file in os.listdir(
                    f"../results/{embedder_folder}/{generator_folder}/{distance_metric}"
                ):
                    if distance_metric.startswith("."):
                        continue

                    results.append(get_results_from_result_file(
                        f"../results/{embedder_folder}/{generator_folder}/{distance_metric}/{result_file}"
                    ))

with open(f"../responses/result_eval.json", "w") as result_file:
    json.dump(results, result_file)

results_df = pd.DataFrame(results)
multi_index_results_df = results_df.set_index(['embedder', 'generator', 'distance_metric']).sort_index()
multi_index_results_df

context_utilization  \
embedder                       generator                     distance_metric                        
BM25                           Ministral-3-14B-Instruct-2512 nan                              1.0   
                                                             nan                              1.0   
                                                             nan                              1.0   
                                                             nan                              1.0   
                                                             nan                              1.0   
...                                                                                           ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                        0.0   
                                                             manhattan                        0.0   
                                                             manhattan                        0.0   
                                                             manhattan                        0.0   
                                                             manhattan                        0.0   

                                                                              answer_relevancy  \
embedder                       generator                     distance_metric                     
BM25                           Ministral-3-14B-Instruct-2512 nan                      0.258085   
                                                             nan                      0.399173   
                                                             nan                      0.000000   
                                                             nan                      0.303063   
                                                             nan                      0.308619   
...                                                                                        ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                0.257794   
                                                             manhattan                0.719062   
                                                             manhattan                0.811106   
                                                             manhattan                0.789209   
                                                             manhattan                0.353063   

                                                                              faithfulness  \
embedder                       generator                     distance_metric                 
BM25                           Ministral-3-14B-Instruct-2512 nan                  0.000000   
                                                             nan                  0.000000   
                                                             nan                  1.000000   
                                                             nan                       NaN   
                                                             nan                       NaN   
...                                                                                    ...   
multilingual-e5-large-instruct Qwen3-30B-A3B-Instruct-2507   manhattan                 NaN   
                                                             manhattan                 NaN   
                                                             manhattan            0.333333   
                                                             manhattan            0.250000   
                                                             manhattan                 NaN   

                                                                                                                    user_prompt  \
embedder                       generator                     distance_metric                                                      
BM25

In [34]:
results_df

,context_utilization,answer_relevancy,faithfulness,user_prompt,embedder,distance_metric,generator,prompt_type,faithfullness
0,0.0,0.000000,0.000000,Wat kun je me vertellen over Rick Kruys?,bge-m3,cosine_similarity,Ministral-3-14B-Instruct-2512,simple_question,NaN
1,0.0,0.000000,NaN,Wat kun je me vertellen over Gemeente Amsterdam?,bge-m3,cosine_similarity,Ministral-3-14B-Instruct-2512,simple_question,NaN
2,0.0,0.338714,NaN,Wat kun je me vertellen over Texel?,bge-m3,cosine_similarity,Ministral-3-14B-Instruct-2512,simple_question,NaN
3,0.0,0.382990,NaN,Wat kun je me vertellen over Bruinvis?,bge-m3,cosine_similarity,Ministral-3-14B-Instruct-2512,simple_question,NaN
4,0.0,0.672624,0.000000,Wat kun je me vertellen over het project Zuida...,bge-m3,cosine_similarity,Ministral-3-14B-Instruct-2512,context_question,NaN
...,...,...,...,...,...,...,...,...,...
1843,0.0,0.257794,NaN,Bruinvis,multilingual-e5-large-instruct,manhattan,Qwen3-30B-A3B-Instruct-2507,single_term,NaN
1844,0.0,0.719062,NaN,Wat kun je me vertellen over Trekkertrek?,multilingual-e5-large-instruct,manhattan,Qwen3-30B-A3B-Instruct-2507,simple_question,NaN
1845,0.0,0.811106,0.333333,Ik ben een journalist uit Noord-Holland en ik ...,multilingual-e5-large-instruct,manhattan,Qwen3-30B-A3B-Instruct-2507,journalist_question,NaN
1846,0.0,0.789209,0.250000,Wat kun je me vertellen over de Gemeente Amste...,multilingual-e5-large-instruct,manhattan,Qwen3-30B-A3B-Instruct-2507,context_question,NaN


## Initial results

In [14]:
metrics = ['mean', 'std']

grouped = results_df.groupby(['embedder', 'generator', 'distance_metric'], dropna=False).agg({'context_utilization' : metrics,
                                                                                              'answer_relevancy' : metrics})
grouped

context_utilization  \
                                                                                                       mean   
embedder                       generator                              distance_metric                         
BM25                           Ministral-3-14B-Instruct-2512          None                         0.847588   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 None                         0.619792   
                               Qwen3-30B-A3B-Instruct-2507            None                         0.750992   
Qwen3-Embedding-8B             Ministral-3-14B-Instruct-2512          cosine_similarity            0.074612   
                                                                      manhattan                    0.102713   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity            0.000000   
                                                                      manhattan                    0.019380   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity            0.044061   
                                                                      manhattan                    0.055882   
bge-m3                         Ministral-3-14B-Instruct-2512          cosine_similarity            0.105364   
                                                                      manhattan                    0.329502   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity            0.000000   
                                                                      manhattan                    0.076705   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity            0.043103   
                                                                      manhattan                    0.255682   
multilingual-e5-large-instruct Ministral-3-14B-Instruct-2512          cosine_similarity            0.088294   
                                                                      manhattan                    0.225379   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity            0.000000   
                                                                      manhattan                    0.035441   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity            0.045977   
                                                                      manhattan                    0.212644   

                                                                                                   \
                                                                                              std   
embedder                       generator                              distance_metric               
BM25                           Ministral-3-14B-Instruct-2512          None               0.287192   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 None               0.426332   
                               Qwen3-30B-A3B-Instruct-2507            None               0.382318   
Qwen3-Embedding-8B             Ministral-3-14B-Instruct-2512          cosine_similarity  0.243889   
                                                                      manhattan          0.238348   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity  0.000000   
                                                                      manhattan          0.109719   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity  0.202471   
                                                                      manhattan          0.182478   
bge-m3                         Ministral-3-14B-Instruct-2512          cosine_similarity  0.295438   
                                                                      manhattan          0.382285   
         

## ANOVA

In [12]:
# Handle missing values in categorical variables (e.g., BM25 doesn't have a distance metric)
results_df['distance_metric'] = results_df['distance_metric'].fillna('None')

metrics = ['answer_relevancy', 'context_utilization']

for metric in metrics:
    # Handle missing values in your target variable if necessary
    anova_df = results_df.dropna(subset=[metric])

    # Define and fit the Ordinary Least Squares (OLS) model
    # Wrap categorical variables in C() to tell statsmodels they are factors
    formula = f'{metric} ~ C(embedder) + C(generator) + C(distance_metric) + C(prompt_type)'
    model = ols(formula, data=anova_df).fit()

    # Perform a Type II ANOVA (recommended for unbalanced/unequal group sizes)
    anova_table = sm.stats.anova_lm(model, typ=2)

    print(f'ANOVA of {metric}')
    
    # Display the results
    print(anova_table)
    print()

ANOVA of answer_relevancy
                        sum_sq      df          F        PR(>F)
C(embedder)           0.000878     3.0   0.004204  9.483094e-01
C(generator)          8.480193     2.0  60.878569  2.618001e-26
C(distance_metric)    0.000586     2.0   0.004204  9.483094e-01
C(prompt_type)        4.817319     3.0  23.055411  1.209448e-14
Residual            125.158222  1797.0        NaN           NaN

ANOVA of context_utilization
                        sum_sq      df          F        PR(>F)
C(embedder)           0.001508     3.0   0.006916  9.931080e-01
C(generator)          6.283849     2.0  43.214557  4.796167e-19
C(distance_metric)    0.001006     2.0   0.006916  9.931080e-01
C(prompt_type)        0.407383     3.0   1.867739  1.330161e-01
Residual            127.234162  1750.0        NaN           NaN



/Users/maritvandenhelder/miniconda3/envs/thesis/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 1
  warnings.warn('covariance of constraints does not have full '
/Users/maritvandenhelder/miniconda3/envs/thesis/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 2, but rank is 1
  warnings.warn('covariance of constraints does not have full '
/Users/maritvandenhelder/miniconda3/envs/thesis/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
  warnings.warn('covariance of constraints does not have full '


## Paired t-test

In [58]:
# Merge on the prompt and generator
merged_df = pd.merge(
    benchmark_df[['user_prompt', 'answer_relevancy', 'generator']], 
    results_df[['embedder', 'generator', 'distance_metric', 'user_prompt', 'answer_relevancy']], 
    on=['user_prompt', 'generator'], 
    suffixes=('_baseline', '_pipeline')
).dropna()  
merged_df['answer_relevancy_difference'] = merged_df['answer_relevancy_pipeline'] - merged_df['answer_relevancy_baseline']
merged_df

,user_prompt,answer_relevancy_baseline,generator,embedder,distance_metric,answer_relevancy_pipeline,answer_relevancy_difference
0,Wat kun je me vertellen over Rick Kruys?,0.258086,Ministral-3-14B-Instruct-2512,bge-m3,cosine_similarity,0.000000,-2.580857e-01
1,Wat kun je me vertellen over Rick Kruys?,0.258086,Ministral-3-14B-Instruct-2512,bge-m3,manhattan,0.000000,-2.580857e-01
2,Wat kun je me vertellen over Rick Kruys?,0.258086,Ministral-3-14B-Instruct-2512,Qwen3-Embedding-8B,cosine_similarity,0.264123,6.036963e-03
3,Wat kun je me vertellen over Rick Kruys?,0.258086,Ministral-3-14B-Instruct-2512,Qwen3-Embedding-8B,manhattan,0.287151,2.906492e-02
4,Wat kun je me vertellen over Rick Kruys?,0.258086,Ministral-3-14B-Instruct-2512,BM25,None,0.258085,-8.097529e-07
...,...,...,...,...,...,...,...
1843,Wat kun je me vertellen over Bruinvissen in No...,0.259636,Qwen3-30B-A3B-Instruct-2507,Qwen3-Embedding-8B,cosine_similarity,0.252758,-6.878405e-03
1844,Wat kun je me vertellen over Bruinvissen in No...,0.259636,Qwen3-30B-A3B-Instruct-2507,Qwen3-Embedding-8B,manhattan,0.412718,1.530819e-01
1845,Wat kun je me vertellen over Bruinvissen in No...,0.259636,Qwen3-30B-A3B-Instruct-2507,BM25,None,0.000000,-2.596360e-01
1846,Wat kun je me vertellen over Bruinvissen in No...,0.259636,Qwen3-30B-A3B-Instruct-2507,multilingual-e5-large-instruct,cosine_similarity,0.000000,-2.596360e-01


In [59]:
# Define a custom function to calculate the p-value
def p_value(series):
    # Drop NaNs to prevent errors in the t-test
    clean_series = series.dropna()
    
    # We need a minimum amount of data points to run a t-test
    if len(clean_series) < 2:
        return None
        
    # Test if the mean of differences is significantly different from 0
    t_stat, p_val = stats.ttest_1samp(clean_series, popmean=0)
    return p_val

metrics = ['mean', 'std', p_value]

merged_grouped = merged_df.groupby(['embedder', 'generator', 'distance_metric'], dropna=False).agg({'answer_relevancy_difference' : metrics})
merged_grouped

answer_relevancy_difference  \
                                                                                                               mean   
embedder                       generator                              distance_metric                                 
BM25                           Ministral-3-14B-Instruct-2512          None                                -0.043291   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 None                                -0.061551   
                               Qwen3-30B-A3B-Instruct-2507            None                                 0.065698   
Qwen3-Embedding-8B             Ministral-3-14B-Instruct-2512          cosine_similarity                   -0.088951   
                                                                      manhattan                            0.002913   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity                    0.086037   
                                                                      manhattan                            0.097593   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity                    0.205534   
                                                                      manhattan                            0.274270   
bge-m3                         Ministral-3-14B-Instruct-2512          cosine_similarity                   -0.047508   
                                                                      manhattan                           -0.061215   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity                    0.105881   
                                                                      manhattan                            0.032492   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity                    0.219268   
                                                                      manhattan                            0.222170   
multilingual-e5-large-instruct Ministral-3-14B-Instruct-2512          cosine_similarity                   -0.056957   
                                                                      manhattan                            0.035150   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity                    0.085171   
                                                                      manhattan                            0.073733   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity                    0.150735   
                                                                      manhattan                            0.210146   

                                                                                                   \
                                                                                              std   
embedder                       generator                              distance_metric               
BM25                           Ministral-3-14B-Instruct-2512          None               0.187085   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 None               0.301841   
                               Qwen3-30B-A3B-Instruct-2507            None               0.254321   
Qwen3-Embedding-8B             Ministral-3-14B-Instruct-2512          cosine_similarity  0.290883   
                                                                      manhattan          0.264987   
                               NVIDIA-Nemotron-3-Super-120B-A12B-BF16 cosine_similarity  0.286916   
                                                                      manhattan          0.315844   
                               Qwen3-30B-A3B-Instruct-2507            cosine_similarity  0.344460   
                                                                      manhattan          0.346025   
bge-m3             